# Tái lập nghiên cứu RAG trên Colab
Notebook này được chuẩn bị cho protocol `research_v1`. **Chưa được xác minh trên một phiên Colab thật chỉ bằng việc tạo notebook.** Chọn runtime GPU, chạy tuần tự và giữ log/metrics cuối. Không gọi run thiếu bước là PASS.

Hai lựa chọn: `rebuild_and_train` dựng nguồn có revision pin rồi train cả hai T5; `evaluate_frozen` dùng dataset/model bundles đã được sao chép. Mọi phép chấm dùng cùng QA/corpus/model protocol; đầu ra test chỉ được xem sau khi cấu hình đã khóa.


In [ ]:
from pathlib import Path
import subprocess, sys, zipfile, json, os, hashlib
PROJECT_DIR = Path('/content/RAG')
SOURCE_ZIP = Path('/content/research_source.zip')
MODE = 'rebuild_and_train'  # hoặc 'evaluate_frozen'
def extract_bundle(path, kind):
    assert path.exists(), f'Cần upload {path.name} vào /content.'
    with zipfile.ZipFile(path) as archive:
        metadata = json.loads(archive.read('bundle_manifest.json'))
        assert metadata['kind'] == kind, 'Sai loại bundle.'
        for name in archive.namelist():
            target = (Path('/content') / name).resolve()
            assert target.is_relative_to(Path('/content').resolve()), 'Archive path escapes /content'
        for relative, expected in metadata['files_sha256'].items():
            member = 'RAG/' + relative
            hasher = hashlib.sha256()
            with archive.open(member) as stream:
                while block := stream.read(8 * 1024 * 1024):
                    hasher.update(block)
            assert hasher.hexdigest() == expected, f'Bundle checksum mismatch: {relative}'
            target = Path('/content') / member
            if target.exists():
                hasher = hashlib.sha256()
                with target.open('rb') as stream:
                    while block := stream.read(8 * 1024 * 1024):
                        hasher.update(block)
                assert hasher.hexdigest() == expected, f'Giữ file hiện có khác bundle: {relative}; dùng VM/thư mục sạch.'
        archive.extractall('/content')
    print('Bundle hashes PASS:', path.name)
if not PROJECT_DIR.exists():
    extract_bundle(SOURCE_ZIP, 'source')
assert (PROJECT_DIR / 'experiments/research_train.py').exists()
if MODE == 'evaluate_frozen':
    extract_bundle(Path('/content/research_data.zip'), 'data')
    extract_bundle(Path('/content/research_models.zip'), 'models')
os.chdir(PROJECT_DIR)
def run(*args):
    subprocess.run([sys.executable, *args], check=True, cwd=PROJECT_DIR)
print('Project:', PROJECT_DIR, 'mode:', MODE)


## Môi trường
Dùng GPU miễn phí nếu được cấp. Không mua hoặc thuê tài nguyên bằng notebook này. Torch CUDA phù hợp có sẵn sẽ được giữ nếu đáp ứng yêu cầu; package versions và GPU thực tế được ghi lại. Nếu Colab yêu cầu restart sau cài đặt, restart và chạy lại cell cấu hình trước khi tiếp tục.


In [ ]:
run('-m', 'pip', 'install', '-r', 'requirements/research.txt', '-r', 'requirements/test.txt')
import torch, platform, importlib.metadata
print('Python:', platform.python_version())
print('Torch:', torch.__version__, 'CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Cần runtime GPU cho full training trong ngân sách thời gian.'
print('GPU:', torch.cuda.get_device_name(0))
print('BF16:', torch.cuda.is_bf16_supported())
# Train dùng BF16 khi được hỗ trợ; GPU không hỗ trợ dùng FP32 và ghi rõ precision.
run('-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_research*.py', '-v')
run('-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_triviaqa_metrics.py', '-v')


## Dựng lại corpus hoặc dùng artifact đóng băng
Gói source chứa manifest nghiên cứu, model revisions và descriptor hash theo record để tránh khác biệt xuống dòng Windows/Linux. Dựng vào thư mục mới; không ghi đè corpus đã chốt. Chế độ frozen cần upload thêm `research_data.zip` và `research_models.zip` vào `/content`; cell đầu tự kiểm tra SHA và giải nén. Dùng VM/thư mục sạch nếu đã có artifacts từ một run khác.


In [ ]:
if MODE == 'rebuild_and_train':
    DATA_DIR = Path('data/research_colab_v1')
    if not (DATA_DIR / 'manifest.json').exists():
        run('-m', 'experiments.research_data', '--reference-manifest', 'data/research_v1/manifest.json',
            '--output', str(DATA_DIR), '--build-index')
    else:
        status = json.loads((DATA_DIR / 'manifest.json').read_text())['status']
        if status != 'complete':
            run('-m', 'experiments.research_data', '--reference-manifest', 'data/research_v1/manifest.json',
                '--output', str(DATA_DIR), '--build-index', '--resume')
else:
    assert MODE == 'evaluate_frozen'
    DATA_DIR = Path('data/research_v1')
run('-m', 'scripts.check_research', '--data-dir', str(DATA_DIR))
print(json.loads((DATA_DIR / 'manifest.json').read_text())['counts'])


## Train hai hệ có kiểm soát
Chạy lần lượt để dùng chung GPU. Mỗi hệ dùng cùng QA IDs đã nghiệm thu và base T5; một hệ context rỗng, một hệ context có bằng chứng. Chọn best checkpoint theo dev loss. Có thể dùng `--resume` với checkpoint chưa hoàn tất, cùng source/config/data; dữ liệu và checkpoint cần được lưu ra Drive nếu muốn giữ qua vòng đời VM.


In [ ]:
if MODE == 'rebuild_and_train':
    for system in ('closed_book', 'rag'):
        model_root = Path('checkpoints/research_v1') / system
        manifest_path = model_root / 'manifest.json'
        if manifest_path.exists() and json.loads(manifest_path.read_text())['status'] == 'complete':
            continue
        args = ['-m', 'experiments.research_train', '--system', system, '--data-dir', str(DATA_DIR)]
        if model_root.exists():
            args += ['--resume']
        run(*args)
for system in ('closed_book', 'rag'):
    run('-m', 'scripts.check_research', '--data-dir', str(DATA_DIR),
        '--model-run', f'checkpoints/research_v1/{system}', '--download-base')


## Dev trước, khóa protocol rồi test
Không chọn phương pháp theo test. Dense, Hybrid, hai baseline và context-ablation dùng cùng cấu hình generation đã công bố. Tên thư mục run không được tồn tại trước; đổi tên khi thực hiện một run mới có lý do.


In [ ]:
DEV_RUN = Path('results/research_colab_dev')
if not DEV_RUN.exists():
    run('-m', 'experiments.research_evaluate', '--data-dir', str(DATA_DIR),
        '--split', 'dev', '--run-dir', str(DEV_RUN))
print(json.loads((DEV_RUN / 'metrics.json').read_text())['qa'])

if not (DEV_RUN / 'REPORT.md').exists():
    run('-m', 'scripts.report_research', '--run-dir', str(DEV_RUN))


In [ ]:
# Chỉ đổi thành True khi đã khóa phương pháp và sẵn sàng chốt holdout.
RUN_FINAL_TEST = False
if RUN_FINAL_TEST:
    TEST_RUN = Path('results/research_colab_test')
    run('-m', 'experiments.research_evaluate', '--data-dir', str(DATA_DIR),
        '--split', 'test', '--final', '--run-dir', str(TEST_RUN))
    print(json.loads((TEST_RUN / 'metrics.json').read_text())['qa'])
    run('-m', 'scripts.report_research', '--run-dir', str(TEST_RUN))
else:
    print('Test NOT RUN: chưa được yêu cầu trong notebook này.')


## Lưu bằng chứng và đánh giá trên mẫu
Lưu manifests, metrics, predictions nén, train logs và source snapshots. Phiếu `human_review_blinded.csv` cần người chấm và lý do; không đổi agent_self_review thành human. Theo `docs/REVIEW_RUBRIC.md` để chấm từng nguồn. Không báo faithfulness/citation đã hoàn tất khi phiếu còn trống.

Chạy cell dưới để đóng gói kết quả; sau đó tải file từ Files hoặc sao chép vào Drive của bạn. Model và dataset bundles là tùy chọn riêng vì dung lượng lớn. So sánh record hashes và cấu hình trước khi so metrics giữa Windows/Colab; không hứa bitwise giống nhau giữa GPU/dtype khác.


In [ ]:
run('-m', 'scripts.export_research_bundle', '--kind', 'evidence', '--data-dir', str(DATA_DIR),
    '--output', 'outputs/research_colab_evidence.zip')
print('Evidence bundle:', PROJECT_DIR / 'outputs/research_colab_evidence.zip')
# Tuỳ chọn khi cần lưu model/data; kiểm tra dung lượng lưu trữ của bạn trước.
# run('-m', 'scripts.export_research_bundle', '--kind', 'models', '--output', 'outputs/research_models.zip')
